# 07 — NDMM PBMC Deep-Clean: Doublet Cluster Removal & Label Correction
### By [Mansi Singh](mansi.singh@alleninstitute.org), Comp Bio, Allen Institute for Immunology
**Aim:** Remove manually annotated doublet clusters from each L3 cell-type subset and apply label corrections (e.g., Core CD14 monocyte clusters 5/6/8 → Other_DC). Export cleaned per-cell-type objects and combined doublet metadata.


## 1 Setup

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

import os
import re
import glob
from datetime import date

import numpy as np               # Numerical computation
import pandas as pd              # DataFrames
import scanpy as sc              # Single-cell analysis
import hisepy                    # HISE platform utilities

In [ ]:
# Output directory for deep-cleaned objects
out_dir = '../../../data/rna/ndmm-pbmc-celltypes-deepcleaned-objs/'
os.makedirs(out_dir, exist_ok=True)

### 1.1 Helper Functions

In [ ]:
def format_cell_type(cell_type):
    """Convert L3 label to file-safe format: '+' → 'pos', '-' → 'neg', ' ' → '_'."""
    cell_type = re.sub('\\+', 'pos', cell_type)
    cell_type = re.sub('-', 'neg', cell_type)
    cell_type = re.sub(' ', '_', cell_type)
    return cell_type

In [ ]:
def get_filepaths_with_glob(root_path: str, file_regex: str):
    """Return list of file paths matching a glob pattern in root_path."""
    return glob.glob(os.path.join(root_path, file_regex))

In [ ]:
# NOTE: Legacy HISE project store search — not needed for local workflow.
# search_id was used to locate files in the HISE project store.
# The cells below that reference h5ad_uuids / sort_adata_uuid
# are retained for provenance but the main workflow reads from local paths.

In [16]:
ps_df = hisepy.list_files_in_project_store('mildModLongi')
ps_df = ps_df[['id', 'name']]

In [17]:
search_df = ps_df[ps_df['name'].str.contains(search_id)]
search_df = search_df.sort_values('name')

In [18]:
h5ad_df = search_df[search_df['name'].str.contains('.h5ad')]

In [20]:
h5ad_uuids = {}
for i in range(h5ad_df.shape[0]):
    fn = h5ad_df['name'].tolist()[i]
    group_name = re.sub('.+pbmc_', '', fn)
    group_name = re.sub('_init.+', '', group_name)
    h5ad_uuids[group_name] = h5ad_df['id'].tolist()[i]

In [22]:
len(h5ad_uuids)

71

In [38]:
# run once
for uuid in h5ad_uuids.values():
    sort_adata_uuid(uuid, sort_cols = ['AIFI_L3'])

downloading fileID: 06e59640-de3c-45cf-b1bc-4150e29046d8
Files have been successfully downloaded!
downloading fileID: a9b29bba-ec08-456d-b83e-2d08726e6b28
Files have been successfully downloaded!
downloading fileID: c77cfb52-7342-4ab0-87f2-55ff0c1500c2
Files have been successfully downloaded!
downloading fileID: 09ee8060-91ed-4932-813e-723c801183ec
Files have been successfully downloaded!
downloading fileID: 2da73675-0401-4527-96b3-764fbcdfab45
Files have been successfully downloaded!
downloading fileID: 930e3900-7436-4bb9-9a16-257f73fef9d9
Files have been successfully downloaded!
downloading fileID: 3ce92a53-50ab-4fb1-84f5-87c3a7ce94a6
Files have been successfully downloaded!
downloading fileID: 58173c98-4b34-44c4-8887-7c18a985f49c
Files have been successfully downloaded!
downloading fileID: 832021ea-284a-4581-965c-56c21fb1610c
Files have been successfully downloaded!
downloading fileID: 68a3ee87-b387-4cde-8dff-f01715d98dfd
Files have been successfully downloaded!
downloading fileID: 

## 2 Load Harmony-Processed Files & Manual Doublet Annotations

In [ ]:
def extract_celltype(path):
    """Extract cell-type name from harmony-processed h5ad filename."""
    pattern = r'-ndmm-pbmc-(.*?)-celltype-harmony-processed\.h5ad'
    match = re.search(pattern, path)
    if match:
        return match.group(1)
    return None

In [ ]:
# Path to per-cell-type Harmony-processed objects (from NB05)
input_path = "../../../data/rna/ndmm-pbmc-celltypes/"

In [ ]:
# Collect Harmony-processed h5ad files and map cell types to file paths
filenames = get_filepaths_with_glob(input_path, "*harmony-processed.h5ad")
cell_types = [extract_celltype(path) for path in filenames]
file_dict = dict(zip(cell_types, filenames))
print(f"Found {len(file_dict)} cell-type files")

['data/ndmm-pbmc-celltypes_MS/2025-06-22-ndmm-pbmc-gzmb+vd2gdt-celltype-harmony-processed.h5ad',
 'data/ndmm-pbmc-celltypes_MS/2025-06-22-ndmm-pbmc-transitionalbcell-celltype-harmony-processed.h5ad',
 'data/ndmm-pbmc-celltypes_MS/2025-06-22-ndmm-pbmc-gzmk-cd56dimnkcell-celltype-harmony-processed.h5ad',
 'data/ndmm-pbmc-celltypes_MS/2025-06-22-ndmm-pbmc-adaptivenkcell-celltype-harmony-processed.h5ad',
 'data/ndmm-pbmc-celltypes_MS/2025-06-22-ndmm-pbmc-isg+cd56dimnkcell-celltype-harmony-processed.h5ad']

In [ ]:
# Read manual doublet annotation spreadsheet (exported from Google Sheets)
# Contains per-cell-type Leiden clusters identified as doublets during manual review (NB06)
doublet_df = pd.read_csv("../../../data/rna/ndmm-manual-doublet-annotation-qc.csv")
print(f"Annotations for {doublet_df.shape[0]} rows")

(211, 5)

In [ ]:
# Standardize column names
doublet_df.rename(columns={
    'Celltypes (AIFI L3)': 'tidy.aifi_l3',
    'leiden cluster to remove': 'leiden_cluster_remove'
}, inplace=True)

['tidy.aifi_l3', 'leiden_cluster_remove', 'Notes', 'Relabel?', 'Genes']

In [113]:
# Select only the relevant columns: cell type and cluster labels indicating doublets
doublet_df = doublet_df[["tidy.aifi_l3", "leiden_cluster_remove"]]

# Reformat cell type names using a custom formatting function
doublet_df['tidy.aifi_l3'] = [format_cell_type(cell_type) for cell_type in doublet_df['tidy.aifi_l3']]

# Convert cluster labels to integer and then to string, keeping NaN values unchanged
doublet_df['leiden_cluster_remove'] = doublet_df['leiden_cluster_remove'].apply(
    lambda x: str(int(x)) if pd.notna(x) else x
)

doublet_df  # Display the cleaned dataframe

,tidy.aifi_l3,leiden_cluster_remove
0,ASDC,5
1,ASDC,13
2,Activated_memory_B_cell,6
3,Activated_memory_B_cell,4
4,Activated_memory_B_cell,5
5,Adaptive_NK_cell,10
6,Adaptive_NK_cell,9
7,Adaptive_NK_cell,13
8,Adaptive_NK_cell,11
9,BaEoMaP_cell,NaN


In [ ]:
# Exclude clusters 5, 6, 8 from Core CD14 monocyte doublet removal
# These clusters are being RELABELED to Other_DC, not removed as doublets
doublet_df = doublet_df[~(
    (doublet_df['tidy.aifi_l3'] == 'Core_CD14_monocyte') &
    (doublet_df['leiden_cluster_remove'].isin(['5', '6', '8']))
)]

In [129]:
len(doublet_df['tidy.aifi_l3'].unique().tolist())

71

In [130]:
doublet_df

,tidy.aifi_l3,leiden_cluster_remove
0,ASDC,5
1,ASDC,13
2,Activated_memory_B_cell,6
3,Activated_memory_B_cell,4
4,Activated_memory_B_cell,5
5,Adaptive_NK_cell,10
6,Adaptive_NK_cell,9
7,Adaptive_NK_cell,13
8,Adaptive_NK_cell,11
9,BaEoMaP_cell,NaN


### 2.1 Collapse Leiden Clusters per Cell Type

In [ ]:
# Group by cell type and aggregate Leiden clusters to remove into lists
collapsed_df = doublet_df.groupby('tidy.aifi_l3')['leiden_cluster_remove'].agg(
    lambda x: list(x) if not x.isnull().all() else np.nan
).reset_index()

pd.set_option('display.max_rows', None)
collapsed_df

,tidy.aifi_l3,leiden_cluster_remove
0,ASDC,"[5, 13]"
1,Activated_memory_B_cell,"[6, 4, 5]"
2,Adaptive_NK_cell,"[10, 9, 13, 11]"
3,BaEoMaP_cell,NaN
4,C1Qpos_CD16_monocyte,"[4, 12, 13, 14]"
5,CD14pos_cDC2,"[10, 3, 18, 19]"
6,CD27neg_effector_B_cell,"[11, 13, 15]"
7,CD27pos_effector_B_cell,"[8, 13]"
8,CD4_MAIT,"[7, 2]"
9,CD56bright_NK_cell,"[11, 14, 15, 13, 10, 5]"


In [134]:
unique_celltypes = collapsed_df['tidy.aifi_l3'].unique()

In [ ]:
def to_marker_style(cell_type):
    """Convert formatted cell-type name to lowercase marker-style key for dict lookup."""
    cell_type = cell_type.replace('_', '').replace('pos', '+').replace('neg', '-')
    return cell_type.lower()

# Map original cell-type names to marker-style keys used in file_dict
unique_celltypes = collapsed_df['tidy.aifi_l3'].unique()
collapsed_df['marker_style'] = collapsed_df['tidy.aifi_l3'].map(
    {ct: to_marker_style(ct) for ct in unique_celltypes}
)

### 2.2 Build Doublet Cluster Dictionary

In [ ]:
# Build dict: marker_style_key → list of Leiden cluster IDs to remove
doublet_cluster_dict = {}
for _, row in collapsed_df.iterrows():
    key = row['marker_style']
    val = row['leiden_cluster_remove']

    if isinstance(val, list):
        values = [str(int(x)) for x in val if pd.notna(x)]
    elif isinstance(val, str):
        values = val.split(',')
    else:
        values = []

    doublet_cluster_dict[key] = values

print(f"{len(doublet_cluster_dict)} cell types with annotations")

{'asdc': ['5', '13'], 'activatedmemorybcell': ['6', '4', '5'], 'adaptivenkcell': ['10', '9', '13', '11'], 'baeomapcell': [], 'c1q+cd16monocyte': ['4', '12', '13', '14'], 'cd14+cdc2': ['10', '3', '18', '19'], 'cd27-effectorbcell': ['11', '13', '15'], 'cd27+effectorbcell': ['8', '13'], 'cd4mait': ['7', '2'], 'cd56brightnkcell': ['11', '14', '15', '13', '10', '5'], 'cd8mait': ['9', '12', '13', '3'], 'cd8aa': ['6'], 'cd95memorybcell': ['11', '10', '14'], 'clpcell': ['13', '10', '12', '5'], 'cmpcell': ['20', '10', '14'], 'cmcd4tcell': ['11'], 'cmcd8tcell': ['11', '13'], 'corecd14monocyte': ['9', '7', '10', '14', '11'], 'corecd16monocyte': ['11', '13', '8'], 'corememorybcell': ['7', '11'], 'corenaivebcell': ['7', '10', '11'], 'corenaivecd4tcell': ['9', '5', '12'], 'corenaivecd8tcell': ['13', '12'], 'dntcell': ['13', '12', '11'], 'earlymemorybcell': ['5', '4'], 'erythrocyte': ['11', '3', '10', '4', '6', '13', '5', '3', '7', '9'], 'gzmb-cd27-emcd4tcell': ['13', '10'], 'gzmb-cd27+emcd4tcell': [

## 3 Generate Deep-Cleaned Objects per Cell Type

In [ ]:
out_files = []   # Track exported h5ad paths
meta_list = []   # Accumulate per-cell-type metadata DataFrames

for label, file in file_dict.items():
    print("Key:", label)
    print("Value:", file)

    # Read the h5ad file
    adata = sc.read_h5ad(file)
    print(adata)

    # --- Relabel Core CD14 monocyte Leiden clusters 5, 6, 8 → Other_DC ---
    # These clusters express DC markers and were misclassified as monocytes
    if 'Other_DC' not in adata.obs['tidy.aifi_l3'].cat.categories:
        adata.obs['tidy.aifi_l3'] = adata.obs['tidy.aifi_l3'].cat.add_categories(['Other_DC'])

    mask = (
        (adata.obs['tidy.aifi_l3'] == 'Core CD14 monocyte') &
        (adata.obs['leiden'].isin(['5', '6', '8']))
    )
    adata.obs.loc[mask, 'tidy.aifi_l3'] = 'Other_DC'

    # --- Mark doublet clusters based on manual annotation ---
    doublet_clust = doublet_cluster_dict.get(label, [])
    print(f"  Doublet clusters: {doublet_clust}")

    if len(doublet_clust) == 0:
        adata.obs['doublets_manual'] = 'no'
    else:
        adata.obs['doublets_manual'] = [
            'yes' if leiden in doublet_clust else 'no'
            for leiden in adata.obs['leiden']
        ]

    print(pd.crosstab(adata.obs['doublets_manual'], adata.obs['leiden']))

    # Collect metadata
    meta = adata.obs
    meta_list.append(meta)
    meta.to_csv(os.path.join(out_dir, f'ndmm_doublet_meta_{label}_{date.today()}.csv'))

    # Subset to singlets only and export cleaned object
    adata_subset = adata[adata.obs['doublets_manual'] == 'no']
    print(f"  Cells retained: {adata_subset.shape[0]:,}")

    out_file = os.path.join(out_dir, f'ndmm_deepcleaned_{label}_{date.today()}.h5ad')
    adata_subset.write_h5ad(out_file)
    out_files.append(out_file)
    print("_" * 65)

Key: gzmb+vd2gdt
Value: data/ndmm-pbmc-celltypes_MS/2025-06-22-ndmm-pbmc-gzmb+vd2gdt-celltype-harmony-processed.h5ad
AnnData object with n_obs × n_vars = 9351 × 2845
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'seurat_pbmc_type', 'seurat_pbmc_type_score', 'umap_1', 'umap_2', 'well_id', 'aifi_l1', 'aifi_l2', 'aifi_l3', 'predicted_doublet', 'doublet_score', 'sample.sampleKitGuid', 'sample.visitDetails', 'sample.visitName', 'sample.drawDate', 'sample.daysSinceFirstVisit', 'sample.diseaseStatesRecordedAtVisit', 'subject.biologicalSex', 'subject.birthYear', 'subject.ethnicity', 'subject.partnerCode', 'subject.race', 'subject.subjectGuid', 'cohort.cohortGuid', 'manual.time_stamp', 'manual.tissue', 'manual.disease_condition', 'manual.response', 'manual.response_type', 'manual.extracted_name', 'manual.file_paths', 'manual.batch_id', 'manual.c

/home/workspace/environment/ndmm-scrna/lib/python3.11/site-packages/anndata/_core/anndata.py:1146: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


Key: transitionalbcell
Value: data/ndmm-pbmc-celltypes_MS/2025-06-22-ndmm-pbmc-transitionalbcell-celltype-harmony-processed.h5ad
AnnData object with n_obs × n_vars = 30546 × 2876
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'seurat_pbmc_type', 'seurat_pbmc_type_score', 'umap_1', 'umap_2', 'well_id', 'aifi_l1', 'aifi_l2', 'aifi_l3', 'predicted_doublet', 'doublet_score', 'sample.sampleKitGuid', 'sample.visitDetails', 'sample.visitName', 'sample.drawDate', 'sample.daysSinceFirstVisit', 'sample.diseaseStatesRecordedAtVisit', 'subject.biologicalSex', 'subject.birthYear', 'subject.ethnicity', 'subject.partnerCode', 'subject.race', 'subject.subjectGuid', 'cohort.cohortGuid', 'manual.time_stamp', 'manual.tissue', 'manual.disease_condition', 'manual.response', 'manual.response_type', 'manual.extracted_name', 'manual.file_paths', 'manual.batch_i

/home/workspace/environment/ndmm-scrna/lib/python3.11/site-packages/anndata/_core/anndata.py:1146: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


Key: gzmk-cd56dimnkcell
Value: data/ndmm-pbmc-celltypes_MS/2025-06-22-ndmm-pbmc-gzmk-cd56dimnkcell-celltype-harmony-processed.h5ad
AnnData object with n_obs × n_vars = 180135 × 1761
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'seurat_pbmc_type', 'seurat_pbmc_type_score', 'umap_1', 'umap_2', 'well_id', 'aifi_l1', 'aifi_l2', 'aifi_l3', 'predicted_doublet', 'doublet_score', 'sample.sampleKitGuid', 'sample.visitDetails', 'sample.visitName', 'sample.drawDate', 'sample.daysSinceFirstVisit', 'sample.diseaseStatesRecordedAtVisit', 'subject.biologicalSex', 'subject.birthYear', 'subject.ethnicity', 'subject.partnerCode', 'subject.race', 'subject.subjectGuid', 'cohort.cohortGuid', 'manual.time_stamp', 'manual.tissue', 'manual.disease_condition', 'manual.response', 'manual.response_type', 'manual.extracted_name', 'manual.file_paths', 'manual.batc

...

/home/workspace/environment/ndmm-scrna/lib/python3.11/site-packages/anndata/_core/anndata.py:1146: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


Key: naivecd4treg
Value: data/ndmm-pbmc-celltypes_MS/2025-06-22-ndmm-pbmc-naivecd4treg-celltype-harmony-processed.h5ad
AnnData object with n_obs × n_vars = 15268 × 2704
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'seurat_pbmc_type', 'seurat_pbmc_type_score', 'umap_1', 'umap_2', 'well_id', 'aifi_l1', 'aifi_l2', 'aifi_l3', 'predicted_doublet', 'doublet_score', 'sample.sampleKitGuid', 'sample.visitDetails', 'sample.visitName', 'sample.drawDate', 'sample.daysSinceFirstVisit', 'sample.diseaseStatesRecordedAtVisit', 'subject.biologicalSex', 'subject.birthYear', 'subject.ethnicity', 'subject.partnerCode', 'subject.race', 'subject.subjectGuid', 'cohort.cohortGuid', 'manual.time_stamp', 'manual.tissue', 'manual.disease_condition', 'manual.response', 'manual.response_type', 'manual.extracted_name', 'manual.file_paths', 'manual.batch_id', 'manua

/home/workspace/environment/ndmm-scrna/lib/python3.11/site-packages/anndata/_core/anndata.py:1146: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


Key: cd27-effectorbcell
Value: data/ndmm-pbmc-celltypes_MS/2025-06-22-ndmm-pbmc-cd27-effectorbcell-celltype-harmony-processed.h5ad
AnnData object with n_obs × n_vars = 2912 × 3942
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'seurat_pbmc_type', 'seurat_pbmc_type_score', 'umap_1', 'umap_2', 'well_id', 'aifi_l1', 'aifi_l2', 'aifi_l3', 'predicted_doublet', 'doublet_score', 'sample.sampleKitGuid', 'sample.visitDetails', 'sample.visitName', 'sample.drawDate', 'sample.daysSinceFirstVisit', 'sample.diseaseStatesRecordedAtVisit', 'subject.biologicalSex', 'subject.birthYear', 'subject.ethnicity', 'subject.partnerCode', 'subject.race', 'subject.subjectGuid', 'cohort.cohortGuid', 'manual.time_stamp', 'manual.tissue', 'manual.disease_condition', 'manual.response', 'manual.response_type', 'manual.extracted_name', 'manual.file_paths', 'manual.batch_

/home/workspace/environment/ndmm-scrna/lib/python3.11/site-packages/anndata/_core/anndata.py:1146: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


In [ ]:
print(f"Exported {len(out_files)} cleaned h5ad files")

['data/ndmm-pbmc-celltypes_deepcleaned_objs/ndmm_deepcleaned_gzmb+vd2gdt_2025-06-27.h5ad',
 'data/ndmm-pbmc-celltypes_deepcleaned_objs/ndmm_deepcleaned_transitionalbcell_2025-06-27.h5ad',
 'data/ndmm-pbmc-celltypes_deepcleaned_objs/ndmm_deepcleaned_gzmk-cd56dimnkcell_2025-06-27.h5ad',
 'data/ndmm-pbmc-celltypes_deepcleaned_objs/ndmm_deepcleaned_adaptivenkcell_2025-06-27.h5ad',
 'data/ndmm-pbmc-celltypes_deepcleaned_objs/ndmm_deepcleaned_isg+cd56dimnkcell_2025-06-27.h5ad',
 'data/ndmm-pbmc-celltypes_deepcleaned_objs/ndmm_deepcleaned_gzmk+cd27+emcd8tcell_2025-06-27.h5ad',
 'data/ndmm-pbmc-celltypes_deepcleaned_objs/ndmm_deepcleaned_clpcell_2025-06-27.h5ad',
 'data/ndmm-pbmc-celltypes_deepcleaned_objs/ndmm_deepcleaned_baeomapcell_2025-06-27.h5ad',
 'data/ndmm-pbmc-celltypes_deepcleaned_objs/ndmm_deepcleaned_il1b+cd14monocyte_2025-06-27.h5ad',
 'data/ndmm-pbmc-celltypes_deepcleaned_objs/ndmm_deepcleaned_cd8mait_2025-06-27.h5ad',
 'data/ndmm-pbmc-celltypes_deepcleaned_objs/ndmm_deepcleaned_

## 4 Export Combined Doublet Metadata

In [ ]:
# Concatenate all per-cell-type metadata into a single DataFrame
doublet_meta_comb = pd.concat(meta_list)
print(f"Combined metadata: {doublet_meta_comb.shape[0]:,} cells × {doublet_meta_comb.shape[1]} columns")

In [ ]:
# Doublet removal summary
print(f"Unique L3 cell types: {doublet_meta_comb['tidy.aifi_l3'].nunique()}")
print(doublet_meta_comb['doublets_manual'].value_counts())

72

In [ ]:
# Compute overall removal statistics
n_total = doublet_meta_comb.shape[0]
n_singlets = (doublet_meta_comb['doublets_manual'] == 'no').sum()
n_removed = n_total - n_singlets
pct_removed = (n_removed / n_total) * 100
print(f"Cells removed: {n_removed:,} / {n_total:,} ({pct_removed:.1f}%)")
print(f"Cells retained: {n_singlets:,}")

12.451108302368874

In [ ]:
# Cell counts per L3 cell type in final combined metadata
doublet_meta_comb['tidy.aifi_l3'].value_counts()

tidy.aifi_l3
Core CD14 monocyte                      408667
Core naive CD4 T cell                   341020
KLRF1- GZMB+ CD27- EM CD8 T cell        273376
GZMK+ CD27+ EM CD8 T cell               264574
GZMK- CD56dim NK cell                   180135
CM CD4 T cell                           175157
Core naive B cell                       111333
Other_DC                                 83923
GZMB- CD27- EM CD4 T cell                80694
ISG+ CD14 monocyte                       79831
Erythrocyte                              72246
GZMB- CD27+ EM CD4 T cell                71344
IL1B+ CD14 monocyte                      67464
KLRF1+ GZMB+ CD27- EM CD8 T cell         63523
Memory CD4 Treg                          60594
Adaptive NK cell                         60378
Core CD16 monocyte                       59084
KLRF1- GZMB+ CD27- memory CD4 T cell     51265
CD56bright NK cell                       33378
GZMK+ CD56dim NK cell                    33226
Transitional B cell                      30546


In [ ]:
# Export combined metadata as CSV and pickle for downstream notebooks
doublet_meta_comb.to_csv("../../../data/rna/ndmm-pbmc-193samples-doublet-metadata-all-types.csv")
doublet_meta_comb.to_pickle("../../../data/rna/ndmm-pbmc-193samples-doublet-metadata-all-types.pkl")
print(f"Exported combined metadata: {doublet_meta_comb.shape}")

In [191]:
doublet_meta_comb.shape

(2944774, 102)